# 01 — Explore LoCoMo Dataset

Load and inspect the LoCoMo dataset. Understand the conversation structure, QA annotation format, and evidence turn mappings.

In [ ]:
import sys
from pathlib import Path

# Resolve project root regardless of where Jupyter was launched from
_root = Path.cwd()
if not (_root / 'src').exists():
    _root = _root.parent
sys.path.insert(0, str(_root / 'src'))

from ingestor import load_locomo, get_stats

In [ ]:
# Load dataset
conversations = load_locomo(_root / 'data' / 'locomo10.json')
print()
print('=== Dataset Stats ===')
stats = get_stats(conversations)

In [ ]:
# Inspect a single conversation
conv = conversations[0]
print(f'Session: {conv.session_id}')
print(f'Turns: {len(conv.turns)}')
print(f'QA pairs: {len(conv.qa_pairs)}')
print()
print('--- First 5 turns ---')
for t in conv.turns[:5]:
    print(f'  [{t.turn_id}] {t.speaker}: {t.text[:100]}')

print()
print('--- First 3 QA pairs ---')
for qa in conv.qa_pairs[:3]:
    print(f'  Q: {qa.question}')
    print(f'  A: {qa.answer}')
    print(f'  Evidence turns: {qa.evidence_turn_ids}')
    print()

In [ ]:
# Distribution of evidence turn distances (how far back does answering require?)
import matplotlib.pyplot as plt

evidence_distances = []
for conv in conversations:
    for qa in conv.qa_pairs:
        if qa.evidence_turn_ids:
            max_turn = max(t.turn_id for t in conv.turns)
            min_evidence = min(qa.evidence_turn_ids)
            evidence_distances.append(max_turn - min_evidence)

plt.figure(figsize=(8, 4))
plt.hist(evidence_distances, bins=20, edgecolor='black')
plt.xlabel('Turns back to evidence')
plt.ylabel('Count')
plt.title('How far back does answering require?')
plt.tight_layout()
plt.savefig(_root / 'results' / 'evidence_distance_dist.png', dpi=100)
plt.show()
print(f'Median lookback: {sorted(evidence_distances)[len(evidence_distances)//2]} turns')

In [ ]:
# QA category distribution
from collections import Counter

categories = [
    qa.category
    for conv in conversations
    for qa in conv.qa_pairs
    if qa.category
]

if categories:
    cat_counts = Counter(categories)
    print('QA categories:')
    for cat, count in cat_counts.most_common():
        print(f'  {cat}: {count}')
else:
    print('No category annotations found')